In [1]:
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 1.8 MB/s eta 0:00:00


In [2]:
!pip install langchain faiss-cpu pandas requests beautifulsoup4 openai gradio tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.6 MB/s eta 0:00:00


In [3]:
!pip install langchain-text-splitters

!pip install langchain-google-genai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.6 MB/s eta 0:00:00


In [4]:
!pip install langchain-community faiss-cpu langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [5]:
import pandas as pd
import requests ## webscrapping
from bs4 import BeautifulSoup
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

import gradio as gr

In [6]:
OPENROUTER_API_KEY = "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"

os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

In [7]:
response = requests.get("https://fullstackacademy.in/")
soup = BeautifulSoup(response.text, "html.parser")
result = soup.get_text(separator=" ")

In [8]:
from langchain_community.document_loaders import TextLoader

doc = TextLoader("/content/course.txt")
txt = doc.load()

In [9]:
txt

[Document(metadata={'source': '/content/course.txt'}, page_content='course\tduration\ttiming\ttrainer_name\nData Science\t4 months\t7pm-9pm\tVarun Maya\nAI Specialist\t3 months\t8pm-10pm\tRahul Verma\nData Analytics\t3 months\t6pm-7:30pm\tSneha Reddy\nDigital Marketing\t2 months\t7am-9am\tImran Khan\nSOC analyst\t2 months\t9pm-10:30pm\tPooja Sharma\nBusiness Analyst\t3 months\t5pm-6:30pm\tKiran Patel\n\t\t\t\n\t\t\t')]

In [10]:
all_text = str(txt)+ "\n" + result

from langchain_huggingface import HuggingFaceEmbeddings

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunks = text_splitter.split_text(all_text)





In [11]:
chunks

["[Document(metadata={'source': '/content/course.txt'}, page_content='course\\tduration\\ttiming\\ttrainer_name\\nData Science\\t4 months\\t7pm-9pm\\tVarun Maya\\nAI Specialist\\t3 months\\t8pm-10pm\\tRahul Verma\\nData Analytics\\t3 months\\t6pm-7:30pm\\tSneha Reddy\\nDigital Marketing\\t2 months\\t7am-9am\\tImran Khan\\nSOC analyst\\t2 months\\t9pm-10:30pm\\tPooja Sharma\\nBusiness Analyst\\t3 months\\t5pm-6:30pm\\tKiran Patel\\n\\t\\t\\t\\n\\t\\t\\t')]",
 'Full Stack Academy | Gateway To Software Industry \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n Full Stack Academy | Gateway To Software Industry \n \n \n \n \n \n \n   \n \n \n \n \n \n \n \n Menu \n \n   \n \n \n Home \n Courses \n Branches \n \n FSA – Tolichowki \n FSA – Gachibowli \n FSA – Ameerpet \n FSA – Charminar \n \n \n About \n Events \n Placements \n \n Companies \n Successfully Placed Candidates \n Candidates available for Placements \n \n \n Contact \n \n \n \n \n \n \n \n 

In [12]:

embeddings = OpenAIEmbeddings(
    model= "text-embedding-3-small"
)


vectorstore = FAISS.from_texts(chunks, embeddings)

In [13]:
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    model="openai/gpt-oss-120b:free",
    temperature=0.7)

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI


llm1 = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key="AIzaSyBK4pTizLkov58ZHPyJCb0b6dstgQc",

    temperature=0.7)

In [15]:
!pip install langchain-classic

In [16]:


from langchain_classic.chains.retrieval_qa.base import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)


In [17]:
qa_chain

RetrievalQA(verbose=False, combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the user's question.\nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\n----------------\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]), llm=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7a644b28ecc0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7a644b28e360>, root_client=<openai.OpenAI object at 0x7a644b

In [18]:
question = "who is the trainer for data science in full stack academy "
answer = qa_chain.invoke({"query": question})
print(answer['result'])

I’m sorry, but I don’t have that information.


In [19]:
# Ask questions

question = "What is this document about?"
answer = qa_chain.invoke({"query": question})
print(answer['result'])

The document is a description of the training institute’s course catalog. It lists the various programs they offer (such as Data Science, AI Specialist, Data Analytics, Digital Marketing, SOC Analyst, Business Analyst, etc.), along with details like the duration of each course, the class timings, and the names of the trainers. It also includes promotional material—ratings, curriculum highlights, certifications, and testimonials—that showcases the institute’s teaching approach and the outcomes for its students. In short, it’s an overview of the institute’s courses and related information for prospective learners.


In [20]:
def chatbot_interface(question):
    answer = qa_chain.invoke({"query": question})
    return answer['result']

In [21]:
iface = gr.Interface(
    fn=chatbot_interface,
    inputs=gr.Textbox(lines=2, placeholder="Ask a question about the courses or academy..."),
    outputs="text",
    title="Full Stack Academy Chatbot",
    description="Ask any question related to the courses, trainers, or the academy."
)

iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3f7667dcd36f788e3f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
